## Object Detection: IoU, NMS e Detectores

Ao final desta aula, você será capaz de:

1. Carregar e usar modelos pré-treinados de detecção (YOLO, Faster R-CNN, Mask R-CNN) em PyTorch.
2. Implementar **IoU** (Intersection over Union) do zero.
3. Implementar **Non-Maximum Suppression (NMS)** do zero.
4. Calcular **Precision, Recall e Average Precision (AP)**.
5. Visualizar **bounding boxes** e **máscaras** de instância.
6. Comparar **YOLO vs. Faster R-CNN vs. Mask R-CNN** em velocidade e qualidade.


## Setup

Usaremos o `torchvision` (Faster R-CNN, Mask R-CNN) e o YOLOv5 via `torch.hub`. Todos os modelos vêm pré-treinados no COCO (80 classes).

In [1]:
# Caso esteja rodando fora do Colab, descomente:
# !pip install torch torchvision matplotlib pillow pandas --quiet

# Força o matplotlib a renderizar inline no notebook
%matplotlib inline

import time
import warnings
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from PIL import Image

import torch
import torchvision
from torchvision import transforms
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    maskrcnn_resnet50_fpn,
)

warnings.filterwarnings('ignore')   # silencia warnings do YOLO sobre urllib3 etc.


ModuleNotFoundError: No module named 'torch'

In [ ]:
# Diagnóstico do ambiente — rode esta célula primeiro
import sys, torch, torchvision

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Matplotlib :", plt.get_backend())
print("CUDA disp. :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
else:
    print(">>> SEM GPU. Faster R-CNN e Mask R-CNN ficam lentos demais na CPU.")
    print(">>> Ambiente de execução > Alterar tipo de ambiente > GPU (T4)")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device     :", device)


> **Os plots não aparecem?** Isso costuma acontecer em ambientes onde o matplotlib não inicializou o backend inline corretamente. Tente:
>
> 1. Rodar a célula `%matplotlib inline` **isolada** primeiro, depois o resto.
> 2. Em Colab: `Runtime → Restart runtime` e rodar tudo de novo.
> 3. Em VSCode/Jupyter local: confirme que está usando um kernel Python real (não "Python: Run").

### Revisão rápida — As 5 tarefas de Computer Vision

| Tarefa | O que faz | Saída |
|--------|-----------|-------|
| **Image Classification** | Atribui um rótulo à imagem | `"cat"` |
| **Object Localization** | Localiza **um** objeto | 1 bounding box + classe |
| **Object Detection** | Localiza **múltiplos** objetos | N bboxes + N classes |
| **Semantic Segmentation** | Rotula cada pixel (sem distinguir instâncias) | máscara por classe |
| **Instance Segmentation** | Rotula pixels + separa cada instância | máscara por objeto |

**Nesta aula:** focamos em **Detection** (YOLO, Faster R-CNN) e **Instance Segmentation** (Mask R-CNN).

In [ ]:
# Função auxiliar para baixar uma imagem de teste
def load_image_from_url(url):
    response = requests.get(url, timeout=10)
    return Image.open(BytesIO(response.content)).convert('RGB')

# Imagem de teste do COCO (clássica: 2 gatos no sofá)
IMG_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
img = load_image_from_url(IMG_URL)

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Imagem de teste — vamos detectar os objetos!')
plt.show()
print(f'Tamanho: {img.size}')

## Parte 1: IoU — Intersection over Union (8 min)

O **IoU** mede a sobreposição entre dois bounding boxes. É a métrica central de avaliação em detecção.

$$IoU = \frac{\text{Área de Interseção}}{\text{Área de União}}$$

- $IoU = 0$ → boxes não se sobrepõem
- $IoU = 1$ → boxes idênticos
- Geralmente: $IoU \geq 0.5$ é considerado uma boa detecção

Cada box é representado como `[x1, y1, x2, y2]` (canto superior esquerdo, canto inferior direito).

### Implementando IoU do zero

Acompanhe o código abaixo passo a passo — cada bloco numerado corresponde a um termo da fórmula.

In [ ]:
def compute_iou(box_a, box_b):
    """
    Calcula IoU entre dois bounding boxes.
    Cada box: [x1, y1, x2, y2]
    """
    # 1) Coordenadas da interseção
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    # 2) Área de interseção (se não houver overlap, retorna 0)
    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h

    # 3) Áreas individuais
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    # 4) União = soma - interseção
    union_area = area_a + area_b - inter_area

    if union_area == 0:
        return 0.0
    return inter_area / union_area

# Teste rápido
box_gt   = [50, 50, 150, 150]   # ground truth
box_pred = [70, 70, 170, 170]   # predição
iou = compute_iou(box_gt, box_pred)
print(f'IoU = {iou:.4f}')
# Conferindo: interseção = 80x80 = 6400; união = 10000 + 10000 - 6400 = 13600
# IoU esperado = 6400/13600 ~ 0.4706
assert abs(iou - 0.4706) < 0.01, 'Algo está errado!'
print('IoU implementado corretamente.')

In [ ]:
# Vamos visualizar diferentes valores de IoU
def plot_two_boxes(box_a, box_b, ax, title):
    ax.add_patch(patches.Rectangle((box_a[0], box_a[1]),
                                    box_a[2]-box_a[0], box_a[3]-box_a[1],
                                    fill=False, edgecolor='green', linewidth=2.5,
                                    label='GT'))
    ax.add_patch(patches.Rectangle((box_b[0], box_b[1]),
                                    box_b[2]-box_b[0], box_b[3]-box_b[1],
                                    fill=False, edgecolor='red', linewidth=2.5,
                                    label='Pred'))
    iou = compute_iou(box_a, box_b)
    ax.set_xlim(0, 250); ax.set_ylim(250, 0)
    ax.set_title(f'{title}\nIoU = {iou:.2f}')
    ax.set_aspect('equal'); ax.legend(loc='upper right')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_two_boxes([50, 50, 150, 150], [55, 55, 145, 145], axes[0], 'Alinhamento quase perfeito')
plot_two_boxes([50, 50, 150, 150], [100, 100, 200, 200], axes[1], 'Sobreposição parcial')
plot_two_boxes([50, 50, 100, 100], [150, 150, 200, 200], axes[2], 'Sem sobreposição')
plt.tight_layout(); plt.show()

### Versão vetorizada (para uso prático com tensores)

Na prática, calculamos IoU entre **muitas** caixas de uma vez. Veja a versão vetorizada:

In [ ]:
def compute_iou_batch(boxes_a, boxes_b):
    """
    Calcula IoU entre todos os pares de boxes_a (N,4) e boxes_b (M,4).
    Retorna tensor (N, M).
    """
    boxes_a = torch.as_tensor(boxes_a, dtype=torch.float32)
    boxes_b = torch.as_tensor(boxes_b, dtype=torch.float32)

    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])

    # Broadcasting: top-left e bottom-right da interseção
    lt = torch.max(boxes_a[:, None, :2], boxes_b[None, :, :2])  # (N, M, 2)
    rb = torch.min(boxes_a[:, None, 2:], boxes_b[None, :, 2:])  # (N, M, 2)

    wh = (rb - lt).clamp(min=0)                # (N, M, 2)
    inter = wh[:, :, 0] * wh[:, :, 1]          # (N, M)

    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union.clamp(min=1e-6)

# Teste
boxes_a = [[50, 50, 150, 150], [200, 200, 300, 300]]
boxes_b = [[70, 70, 170, 170], [220, 220, 280, 280], [0, 0, 50, 50]]
iou_matrix = compute_iou_batch(boxes_a, boxes_b)
print('Matriz IoU (linhas = boxes_a, colunas = boxes_b):')
print(iou_matrix)

## Parte 2: Non-Maximum Suppression (8 min)

Detectores como YOLO geram **milhares** de caixas por imagem (no YOLOv3: ~10.647). A maior parte é redundante.

**Ideia do NMS:** para cada classe, mantenha a caixa com maior score e descarte qualquer outra com IoU alto (sobreposição). Repita.

### Algoritmo

Para cada classe:
1. Descarte caixas com score $\leq 0.6$
2. Selecione a caixa com maior score $p_c$ → vira predição
3. Descarte demais caixas com $IoU \geq 0.5$ em relação à anterior
4. Repita 2-3 com as remanescentes

In [ ]:
def non_max_suppression(boxes, scores, score_thresh=0.6, iou_thresh=0.5):
    """
    boxes:  (N, 4) — [x1, y1, x2, y2]
    scores: (N,)   — confiança de cada box
    Retorna índices das caixas mantidas.
    """
    boxes  = torch.as_tensor(boxes, dtype=torch.float32)
    scores = torch.as_tensor(scores, dtype=torch.float32)

    # 1) filtro por score
    keep_mask = scores > score_thresh
    idxs = torch.where(keep_mask)[0]
    if len(idxs) == 0:
        return torch.empty(0, dtype=torch.long)

    boxes_f  = boxes[idxs]
    scores_f = scores[idxs]

    # 2) ordena por score decrescente
    order = scores_f.argsort(descending=True)

    kept = []
    while len(order) > 0:
        # pega a de maior score
        i = order[0].item()
        kept.append(idxs[i].item())

        if len(order) == 1:
            break

        # IoU da escolhida vs. demais
        rest = order[1:]
        ious = compute_iou_batch(boxes_f[i].unsqueeze(0), boxes_f[rest])[0]

        # mantém só as com baixo IoU
        order = rest[ious < iou_thresh]

    return torch.tensor(kept, dtype=torch.long)

# Teste: 4 caixas detectadas, 3 são para o mesmo objeto
test_boxes = torch.tensor([
    [100, 100, 200, 200],   # box A
    [105, 105, 205, 205],   # quase idêntica à A (overlap alto)
    [110, 110, 210, 210],   # idem
    [300, 300, 400, 400],   # objeto diferente
], dtype=torch.float32)
test_scores = torch.tensor([0.9, 0.85, 0.7, 0.95])

kept_idx = non_max_suppression(test_boxes, test_scores)
print(f'Caixas mantidas: {kept_idx.tolist()}')
print(f'Esperado: [3, 0] — a 3 tem maior score; depois a 0 sobra (sem overlap com 3)')

In [ ]:
# Visualização: antes e depois do NMS
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['red', 'orange', 'purple', 'blue']

for ax, title, indices in [
    (axes[0], 'ANTES do NMS (4 caixas)', range(len(test_boxes))),
    (axes[1], 'DEPOIS do NMS (apenas as melhores)', kept_idx.tolist())
]:
    for i in indices:
        b = test_boxes[i]
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                        fill=False, edgecolor=colors[i], linewidth=2.5))
        ax.text(b[0], b[1]-5, f'{test_scores[i]:.2f}', color=colors[i],
                fontsize=11, fontweight='bold')
    ax.set_xlim(50, 450); ax.set_ylim(450, 50)
    ax.set_title(title); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

### Atalho: o PyTorch já tem NMS pronto e otimizado em CUDA

Na prática, use `torchvision.ops.nms` — mas saber implementá-lo é importante para entender o que acontece por trás.

In [ ]:
from torchvision.ops import nms
kept_torchvision = nms(test_boxes, test_scores, iou_threshold=0.5)
print(f'NMS do torchvision: {kept_torchvision.tolist()}')
print(f'Nosso NMS:          {kept_idx.tolist()}')
# (a ordem pode diferir, mas o conjunto de índices deve coincidir)

## Parte 3: YOLO em ação (10 min)

**YOLO** (You Only Look Once) é a família de detectores **single-stage** mais famosa. A ideia central, do paper de Redmon et al. (2016):

1. Divide a imagem em uma grade `n x n` (slides 25-26).
2. Para cada célula da grade, prevê **B anchor boxes**, cada uma com `[p_c, b_x, b_y, b_h, b_w, c_1, ..., c_C]`.
3. Uma única passada da CNN gera tudo de uma vez (daí "you only look once").
4. Aplica NMS no final para limpar predições redundantes.

No YOLOv3, isso resulta em $(13 \times 13 + 26 \times 26 + 52 \times 52) \times 3 \approx 10.647$ predições por imagem em 3 escalas — daí ser **excelente para objetos de tamanhos diversos**.

Vamos usar o **YOLOv5** (família moderna, mantida pela Ultralytics), que segue os mesmos princípios.

In [ ]:
# O YOLOv5 do torch.hub usa o pacote `ultralytics` para checar dependências;
# instalá-lo antes evita o warning de AutoUpdate na célula seguinte.
!pip install ultralytics --quiet


In [ ]:
# Carregando YOLOv5 via torch.hub — os pesos vêm do repositório oficial.
# Primeiro uso: baixa ~14 MB para yolov5s (small).
# Variantes disponíveis: yolov5n (nano), yolov5s, yolov5m, yolov5l, yolov5x (extra-large)
print('Carregando YOLOv5s...')
yolo = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True, verbose=False)
yolo = yolo.to(device).eval()
print('Modelo pronto.')
print(f'Número de classes: {len(yolo.names)}')
print(f'Algumas classes: {list(yolo.names.values())[:10]}')

In [ ]:
# Inferência YOLO — a API é bem direta
results = yolo(img)
results.print()  # mostra resumo

# Os resultados vêm em um DataFrame pandas:
df = results.pandas().xyxy[0]   # xyxy = formato [x1, y1, x2, y2]
print('\nDetecções:')
print(df)

In [ ]:
# Vamos padronizar a saída para ser igual à do torchvision
# (boxes, labels, scores) para usar a mesma função de plotagem nos dois modelos
def yolo_to_dict(results, score_thresh=0.25):
    df = results.pandas().xyxy[0]
    df = df[df['confidence'] > score_thresh]
    boxes = torch.tensor(df[['xmin', 'ymin', 'xmax', 'ymax']].values, dtype=torch.float32)
    labels = torch.tensor(df['class'].values, dtype=torch.long)
    scores = torch.tensor(df['confidence'].values, dtype=torch.float32)
    return {'boxes': boxes, 'labels': labels, 'scores': scores, 'class_names': df['name'].tolist()}

yolo_result = yolo_to_dict(results, score_thresh=0.5)
print(f'YOLO detectou {len(yolo_result["boxes"])} objetos com score > 0.5')
for box, name, score in zip(yolo_result['boxes'], yolo_result['class_names'], yolo_result['scores']):
    print(f'  - {name:15s} score={score:.3f}')

In [ ]:
def plot_detections(pil_img, boxes, labels_text, scores, title='Detecções'):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(pil_img)
    cmap = plt.get_cmap('tab10')

    for i, (box, label, score) in enumerate(zip(boxes, labels_text, scores)):
        x1, y1, x2, y2 = [float(v) for v in box]
        color = cmap(i % 10)
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                        linewidth=2.5, edgecolor=color, facecolor='none'))
        ax.text(x1, y1-5, f'{label} {float(score):.2f}',
                color='white', fontsize=11, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=2, alpha=0.85))
    ax.axis('off')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_detections(img,
                yolo_result['boxes'],
                yolo_result['class_names'],
                yolo_result['scores'],
                title='YOLOv5s — Object Detection')

### Experimente: efeito do threshold de confiança

À medida que o threshold cai, o **recall sobe** e a **precision tende a cair** (mais detecções,
incluindo falsos positivos). É exatamente o trade-off que a curva PR da Parte 6 desenha.

Atenção a uma pegadinha: o YOLO **já filtra internamente** em `yolo.conf` (default `0.25`), então
filtrar o resultado abaixo desse valor não traz caixa nenhuma a mais. Para varrer a faixa toda,
baixamos `yolo.conf` e refazemos a inferência.


In [ ]:
conf_original = yolo.conf
yolo.conf = 0.01                 # praticamente sem filtro interno
results_todos = yolo(img)

for thr in [0.9, 0.5, 0.25, 0.1, 0.05]:
    r = yolo_to_dict(results_todos, score_thresh=thr)
    print(f'Threshold={thr:<5} -> {len(r["boxes"])} detecções')

yolo.conf = conf_original        # restaura o default


## Parte 4: Faster R-CNN — single-stage vs. two-stage (8 min)

Agora vamos comparar com um **detector two-stage**: Faster R-CNN.

**Diferença fundamental:**
- **YOLO (single-stage):** uma única passada da rede gera caixas e classes simultaneamente. Rápido, ideal para tempo-real.
- **Faster R-CNN (two-stage):** primeiro a RPN gera propostas de regiões; depois uma cabeça classifica + refina cada uma. Mais lento, mas costuma ter melhor mAP em objetos pequenos.

Vamos rodar os dois e comparar **número de detecções e tempo de inferência**.

In [ ]:
# Classes do COCO (necessário para o Faster R-CNN do torchvision)
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
    'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut',
    'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book', 'clock',
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]
print(f'COCO tem {len([c for c in COCO_CLASSES if c != "N/A"])-1} classes válidas.')

In [ ]:
# Carregando Faster R-CNN pré-treinado (no primeiro uso, baixa ~160 MB)
print('Carregando Faster R-CNN...')
frcnn = fasterrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval()
print('Modelo pronto.')

def detect_torchvision(model, pil_image, score_thresh=0.7):
    """Inferência genérica para modelos do torchvision (Faster R-CNN, Mask R-CNN)."""
    transform = transforms.Compose([transforms.ToTensor()])
    img_t = transform(pil_image).to(device)
    with torch.no_grad():
        preds = model([img_t])[0]
    keep = preds['scores'] > score_thresh
    return {
        'boxes':  preds['boxes'][keep].cpu(),
        'labels': preds['labels'][keep].cpu(),
        'scores': preds['scores'][keep].cpu(),
        'masks':  preds['masks'][keep].cpu() if 'masks' in preds else None,
    }

frcnn_result = detect_torchvision(frcnn, img, score_thresh=0.7)
label_names = [COCO_CLASSES[l] for l in frcnn_result['labels']]
plot_detections(img, frcnn_result['boxes'], label_names, frcnn_result['scores'],
                title='Faster R-CNN — Object Detection')

In [ ]:
# Benchmark de velocidade: rodar cada modelo várias vezes e medir tempo.
# Medimos a chamada inteira (pré-processamento + rede + NMS), que é o que
# interessa na prática — não só o forward da CNN.
def benchmark(fn, n_runs=5, warmup=2):
    for _ in range(warmup):
        fn()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_runs):
        fn()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - t0) / n_runs * 1000   # ms por imagem

yolo_ms  = benchmark(lambda: yolo(img))
frcnn_ms = benchmark(lambda: detect_torchvision(frcnn, img, 0.7))

print(f'YOLOv5s        : {yolo_ms:.1f} ms/imagem')
print(f'Faster R-CNN   : {frcnn_ms:.1f} ms/imagem')
print(f'\nYOLO é ~{frcnn_ms/yolo_ms:.1f}x mais rápido nesta máquina.')

In [ ]:
# Comparação lado a lado em uma imagem mais complexa
img2 = load_image_from_url('http://images.cocodataset.org/val2017/000000000785.jpg')

yolo_r2  = yolo_to_dict(yolo(img2), score_thresh=0.5)
frcnn_r2 = detect_torchvision(frcnn, img2, score_thresh=0.7)
frcnn_names2 = [COCO_CLASSES[l] for l in frcnn_r2['labels']]

plot_detections(img2, yolo_r2['boxes'], yolo_r2['class_names'], yolo_r2['scores'],
                title=f'YOLOv5s — {len(yolo_r2["boxes"])} detecções')
plot_detections(img2, frcnn_r2['boxes'], frcnn_names2, frcnn_r2['scores'],
                title=f'Faster R-CNN — {len(frcnn_r2["boxes"])} detecções')

## Parte 5: Mask R-CNN — Instance Segmentation (8 min)

Mask R-CNN estende Faster R-CNN com uma **terceira saída**: uma máscara binária por instância.

**Pipeline:**
1. **RPN** (Region Proposal Network) → gera regiões candidatas
2. **ROIAlign** → redimensiona regiões para tamanho fixo (sem rounding! usa interpolação bilinear)
3. **Heads paralelos:**
   - classificação + bbox refinement (igual Faster R-CNN)
   - **FCN gera a máscara de segmentação**

In [ ]:
print('Carregando Mask R-CNN...')
mask_model = maskrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval()
print('Modelo pronto.')

mask_result = detect_torchvision(mask_model, img2, score_thresh=0.7)
print(f'\n{len(mask_result["boxes"])} instâncias detectadas')
print(f'Shape das máscaras: {mask_result["masks"].shape}')
# (N, 1, H, W) — uma máscara por instância, valores entre 0 e 1

In [ ]:
# Visualização com máscaras + bboxes
def plot_instance_segmentation(pil_img, result, classes, mask_thresh=0.5, title='Mask R-CNN'):
    img_np = np.array(pil_img).copy()
    overlay = img_np.astype(np.float32).copy()
    cmap = plt.get_cmap('tab10')

    # 1) pinta cada máscara
    for i, mask in enumerate(result['masks']):
        binary_mask = (mask[0].numpy() > mask_thresh)
        color = np.array(cmap(i % 10)[:3]) * 255   # RGB
        for c in range(3):
            overlay[:, :, c] = np.where(binary_mask,
                                         0.5 * overlay[:, :, c] + 0.5 * color[c],
                                         overlay[:, :, c])

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(overlay.astype(np.uint8))

    # 2) bboxes por cima
    for i, (box, label, score) in enumerate(zip(result['boxes'], result['labels'], result['scores'])):
        x1, y1, x2, y2 = box.tolist()
        color = cmap(i % 10)
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                        linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, y1-5, f'{classes[label]} {score:.2f}',
                color='white', fontsize=11, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=2, alpha=0.85))
    ax.axis('off'); ax.set_title(title)
    plt.tight_layout(); plt.show()

plot_instance_segmentation(img2, mask_result, COCO_CLASSES,
                            title='Mask R-CNN — Instance Segmentation')

### Comparação dos três modelos lado a lado

Repare: o que diferencia Object Detection (caixa) de Instance Segmentation (máscara pixel-a-pixel) — a caixa é uma aproximação grosseira; a máscara segue o contorno do objeto.

In [ ]:
# YOLO (single-stage) | Faster R-CNN (two-stage) | Mask R-CNN (segmentation)
plot_detections(img2, yolo_r2['boxes'], yolo_r2['class_names'], yolo_r2['scores'],
                title='1) YOLOv5s — Single-stage detection')
plot_detections(img2, frcnn_r2['boxes'], frcnn_names2, frcnn_r2['scores'],
                title='2) Faster R-CNN — Two-stage detection')
plot_instance_segmentation(img2, mask_result, COCO_CLASSES,
                            title='3) Mask R-CNN — Instance segmentation')

## Parte 6: Métricas — Precision, Recall e AP (8 min)

Como avaliar um detector? Para cada predição, com IoU vs. ground truth:
- **TP** (true positive):  $IoU \geq 0.5$ com algum GT da mesma classe
- **FP** (false positive): não bate com nenhum GT
- **FN** (false negative): GT que ficou sem predição

$$Precision = \frac{TP}{TP + FP}, \quad Recall = \frac{TP}{TP + FN}$$

**AP (Average Precision)** = área sob a curva Precision-Recall (versão interpolada).

**mAP** = média de AP sobre todas as classes — é a métrica padrão do COCO.

In [ ]:
def precision_recall(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    """
    Versão simplificada (1 classe): retorna arrays precision/recall conforme
    diminuímos o threshold de score.
    """
    pred_boxes  = torch.as_tensor(pred_boxes,  dtype=torch.float32)
    pred_scores = torch.as_tensor(pred_scores, dtype=torch.float32)
    gt_boxes    = torch.as_tensor(gt_boxes,    dtype=torch.float32)

    n_gt = len(gt_boxes)
    if n_gt == 0:
        return np.array([]), np.array([])

    # ordena preds por score (alto -> baixo)
    order = pred_scores.argsort(descending=True)
    pred_boxes = pred_boxes[order]

    tp = np.zeros(len(pred_boxes))
    fp = np.zeros(len(pred_boxes))
    matched_gt = set()

    for i, pbox in enumerate(pred_boxes):
        ious = compute_iou_batch(pbox.unsqueeze(0), gt_boxes)[0]
        best_iou, best_gt = ious.max(0)
        if best_iou >= iou_thresh and best_gt.item() not in matched_gt:
            tp[i] = 1
            matched_gt.add(best_gt.item())
        else:
            fp[i] = 1

    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)
    precision = cum_tp / (cum_tp + cum_fp + 1e-9)
    recall    = cum_tp / n_gt
    return precision, recall

def compute_ap(precision, recall):
    """AP usando precisão interpolada (definição do PASCAL VOC)."""
    if len(precision) == 0:
        return 0.0
    # acrescenta sentinels
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    # interpolação: precisão máxima à direita
    for i in range(len(mpre)-1, 0, -1):
        mpre[i-1] = max(mpre[i-1], mpre[i])
    # integral
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[i+1] - mrec[i]) * mpre[i+1]))

In [ ]:
# Toy example: 3 ground-truth, 5 predições (algumas certas, algumas erradas)
gt_boxes = [
    [50, 50, 150, 150],
    [200, 200, 300, 300],
    [400, 100, 500, 200],
]
pred_boxes = [
    [52, 48, 148, 152],     # TP (bate com GT 0)
    [205, 198, 295, 305],   # TP (bate com GT 1)
    [410, 110, 490, 190],   # TP (bate com GT 2)
    [600, 600, 700, 700],   # FP (sem GT)
    [55, 55, 145, 145],     # FP (mesmo GT 0 já matched)
]
pred_scores = [0.95, 0.90, 0.80, 0.85, 0.60]

p, r = precision_recall(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5)
ap = compute_ap(p, r)

print(f'Precision: {np.round(p, 3)}')
print(f'Recall:    {np.round(r, 3)}')
print(f'AP @ 0.5 = {ap:.3f}')

In [ ]:
# Curva PR
plt.figure(figsize=(7, 5))
plt.step(r, p, where='post', linewidth=2.5, label=f'AP = {ap:.3f}')
plt.fill_between(r, p, step='post', alpha=0.2)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.xlim(0, 1.05); plt.ylim(0, 1.05)
plt.grid(alpha=0.3); plt.legend()
plt.show()

## Resumo e próximos passos

Nesta aula, vimos hands-on:

- **IoU** — métrica fundamental de sobreposição (do zero + vetorizado)
- **NMS** — como detectores filtram milhares de caixas em poucas
- **YOLOv5** — detector single-stage, rápido, ideal para tempo-real
- **Faster R-CNN** — detector two-stage, mais lento porém preciso
- **Mask R-CNN** — instance segmentation com ROIAlign
- **Precision / Recall / AP** — como medir performance

### Desafios para casa

1. **Compare YOLOv5n vs YOLOv5x**: meça mAP e velocidade. Onde está o trade-off?
2. **Curva PR real**: pegue uma imagem do COCO com anotações ground-truth e plote a curva variando o threshold do score.
3. **Soft-NMS**: pesquise e implemente — em vez de descartar a caixa redundante, diminui o score dela proporcionalmente ao IoU.
4. **Fine-tuning**: treine o YOLOv5 em um dataset customizado (ex: detector de placas de trânsito). A documentação Ultralytics tem um tutorial passo-a-passo.

### Referências

- Redmon et al. (2016) *You Only Look Once: Unified, Real-Time Object Detection*
- Redmon & Farhadi (2018) *YOLOv3: An Incremental Improvement*
- He et al. (2018) *Mask R-CNN*
- Ren et al. (2016) *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*
- Documentação YOLOv5: https://github.com/ultralytics/yolov5



## Avaliação

Deixe seu feedback da aula pelo **formulário linkado na coluna _Feedback_ do
[README do repositório](https://github.com/Erickslb/deep-learning-fgv-2026)** — o que funcionou, o que ficou confuso, o que faltou.

Dúvidas técnicas que podem interessar aos colegas ficam melhor como
[issue](https://github.com/Erickslb/deep-learning-fgv-2026/issues), que todo mundo vê.
